# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

> **⚠️ Execution note:** the code cells below query the gated Hugging Face warehouse release (`FlyRank/internship-warehouse`) using your own `HF_TOKEN`. They have NOT been run yet in this file — run this notebook top-to-bottom in Colab (Runtime → Run all) with your token set as a Colab Secret, so the real outputs get embedded before you commit. Nothing here is a placeholder number; the cells are written to run correctly, they just haven't been executed against your token yet.

## 1. Unit of analysis + time window

**One row = one (client, content item, day)** — a single page's daily performance for a single client, in `fact_content_daily_performance`. That's the grain I build Lane 4's features from; I aggregate up to one row per (client, content item) only when I build a feature frame for a specific window.

**Time window for this contract: `report_date` in March 2026 (`month=2026-03`)** — a mid-panel month, not the final sealed month (`_sample`, June 2026), per the warning on this card: the last month is the natural outcome window for any past→future label, so it's reserved for held-out testing later, never for developing label logic now.

I verify both of these claims with real queries in Section 3 below.

In [ ]:
# No query needed here yet -- Section 3 verifies the grain and window claims above with real queries.
# This cell is intentionally left as a placeholder per the skeleton's own instruction.

## 2. Fields: feature / label / context / excluded

For Lane 4 (CTR / Engagement Opportunity Scoring), sorting the fields I plan to touch:

| Bucket | Fields | Why |
|---|---|---|
| **Feature** | `gsc_impressions`, `gsc_clicks`, `gsc_avg_position` (from `fact_content_daily_performance`); `ga4_sessions`, `ga4_engagement_rate` (same table); `word_count`, `main_intent`, `content_type` (from `dim_content`) | All knowable at or before the moment I'd rank a page — none of them peek into a future window |
| **Label / proxy** | `is_low_ctr` — a page's CTR below the current month's median (derived, same-window proxy, **not** a genuine future outcome) | This is what I'd predict or rank by; it must never also appear as a feature |
| **Context** | `client_hash_id`, `content_hash_id`, `report_date`, `keyword_hash_id`, `url_hash_id` | Used only for joining, grouping, and deduplication — the model never learns from the codes themselves |
| **Excluded** | `health_score`, `priority_score`, `action_type`, `refresh_tier` (product decision flags — not shipped in the release anyway, and never to be rebuilt and fed back in as a feature); any raw query/URL/title text (scrambled before release, and reconstructing it would violate the public-safe rule) | Product flags would let a model just copy FlyRank's existing rule instead of discovering signal; raw text was deliberately never released |

I verify the field-level claims (mainly missingness/availability) with a real query in Section 3.

In [1]:
# No query needed here yet -- the availability/missingness check for these fields
# is the "IS TRUE" query in Section 3.

## 3. Verify it with queries (grain, counts, missing values, windows)

Three required verification queries, plus the five-feature frame and the deliberate leakage trap, all against `month=2026-03` on the real warehouse release. **Run this section in Colab with your `HF_TOKEN`** — every query below is written to run correctly, but the outputs are not embedded in this copy yet.

**Query 1 — grain check.** If the grain from Section 1 is right, grouping by (client, content, day) and filtering for duplicates should return an empty result.

In [2]:
# %pip -q install duckdb huggingface_hub

import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':    f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':     f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':      f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_query_90d':  f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

grain = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) c
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01' AND report_date < DATE '2026-04-01'
    GROUP BY 1,2,3
    HAVING c > 1
    LIMIT 5
""").df()
print('=== QUERY 1: GRAIN CHECK (empty = grain holds) ===')
print(grain)

Paste your Hugging Face READ token (hf_...): ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== QUERY 1: GRAIN CHECK (empty = grain holds) ===
Empty DataFrame
Columns: [client_hash_id, content_hash_id, report_date, c]
Index: []


**Query 2 — row count and date span.** Confirms the March 2026 slice is the size and window I claimed above.

In [3]:
counts = con.sql(f"""
    SELECT COUNT(*) AS row_count, MIN(report_date) AS min_date, MAX(report_date) AS max_date,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           COUNT(DISTINCT content_hash_id) AS n_content
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01' AND report_date < DATE '2026-04-01'
""").df()
print('=== QUERY 2: ROW COUNT / DATE SPAN (March 2026) ===')
print(counts)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== QUERY 2: ROW COUNT / DATE SPAN (March 2026) ===
   row_count   min_date   max_date  n_clients  n_content
0    9841378 2026-03-01 2026-03-31         55     331437


**Query 3 — availability check, using `IS TRUE`.** GSC/GA4 data isn't available for every row (tracking starts at different times per client) — this shows how many rows in the window actually have usable data, not just how many rows exist.

In [4]:
avail = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01' AND report_date < DATE '2026-04-01'
""").df()
print('=== QUERY 3: AVAILABILITY CHECK (IS TRUE) ===')
print(avail)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== QUERY 3: AVAILABILITY CHECK (IS TRUE) ===
   total_rows  gsc_available_rows  ga4_available_rows
0     9841378           3611061.0            413966.0


**Five features, max — each with an "available when?" line:**

1. `avg_position_mar` — average GSC position over March: knowable at month's end, before I'd act on it
2. `impressions_mar` — total GSC impressions over March: same, purely historical by the time I'd use it
3. `ctr_mar` — clicks/impressions for March: derived only from the same historical fields above
4. `sessions_mar` — GA4 sessions over March: knowable at month's end
5. `engagement_rate_mar` — average GA4 engagement rate over March: knowable at month's end

All five are aggregates of a *closed* month — nothing here reaches into April or beyond.

In [8]:
features = con.sql(f"""
    SELECT
        f.client_hash_id, f.content_hash_id,
        AVG(f.gsc_avg_position) AS avg_position_mar,
        SUM(f.gsc_impressions) AS impressions_mar,
        SUM(f.gsc_clicks) AS clicks_mar,
        SUM(f.gsc_clicks) * 1.0 / NULLIF(SUM(f.gsc_impressions), 0) AS ctr_mar,
        SUM(f.ga4_sessions) AS sessions_mar,
        SUM(f.ga4_engaged_sessions) AS engaged_sessions_mar,
        SUM(f.ga4_engaged_sessions) * 1.0 / NULLIF(SUM(f.ga4_sessions), 0) AS engagement_rate_mar
    FROM {TABLES['fact_daily']} f
    WHERE f.month = '2026-03'
      AND f.gsc_data_available IS TRUE
    GROUP BY 1,2
    HAVING SUM(f.gsc_impressions) >= 500
""").df()
print('=== FIVE-FEATURE FRAME (Lane 4, March 2026, impressions>=500) ===')
print(f'shape: {features.shape}')
features.head(8)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== FIVE-FEATURE FRAME (Lane 4, March 2026, impressions>=500) ===
shape: (61924, 9)


,client_hash_id,content_hash_id,avg_position_mar,impressions_mar,clicks_mar,ctr_mar,sessions_mar,engaged_sessions_mar,engagement_rate_mar
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,4.394234,1140.0,2.0,0.001754,0.0,0.0,NaN
1,client_73cda7b4e4f265ea,content_05434271b257bb68,6.320337,1421.0,6.0,0.004222,9.0,0.0,0.0
2,client_73cda7b4e4f265ea,content_d056587ff7faca0c,4.459107,2770.0,16.0,0.005776,3.0,0.0,0.0
3,client_73cda7b4e4f265ea,content_712c365258cee05c,4.950311,6048.0,23.0,0.003803,8.0,0.0,0.0
4,client_73cda7b4e4f265ea,content_8935ed68eca88b01,9.306264,4788.0,10.0,0.002089,14.0,0.0,0.0
5,client_73cda7b4e4f265ea,content_6288c1f7d664fa4b,4.257884,960.0,3.0,0.003125,1.0,0.0,0.0
6,client_73cda7b4e4f265ea,content_0c29bba4e77ab325,3.915574,4650.0,12.0,0.002581,2.0,0.0,0.0
7,client_73cda7b4e4f265ea,content_17680db83b4a16c9,15.048245,1609.0,0.0,0.000000,1.0,0.0,0.0


**The trap.** Adding one label-derived column on purpose — `LEAKY_ctr_bucket`, which is just the CTR itself rounded, almost identical information to the label I'm trying to predict. Watch the score jump toward a suspiciously perfect number, then delete it and keep the honest one.

In [10]:
features['ctr_mar'] = features['ctr_mar'].fillna(0)
features['is_low_ctr'] = (features['ctr_mar'] < features['ctr_mar'].median()).astype(int)

# LEAKY feature -- derived almost directly from the label itself
features['LEAKY_ctr_bucket'] = features['ctr_mar'].round(2)

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

honest_cols = ['avg_position_mar', 'impressions_mar', 'sessions_mar', 'engagement_rate_mar']
leaky_cols = honest_cols + ['LEAKY_ctr_bucket']

model_data = features.dropna(subset=leaky_cols + ['is_low_ctr'])
X_honest, X_leaky, y = model_data[honest_cols], model_data[leaky_cols], model_data['is_low_ctr']

Xh_tr, Xh_te, y_tr, y_te = train_test_split(X_honest, y, test_size=0.25, random_state=42, stratify=y)
Xl_tr, Xl_te, _, _       = train_test_split(X_leaky,  y, test_size=0.25, random_state=42, stratify=y)

m_honest = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(Xh_tr, y_tr)
m_leaky  = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(Xl_tr, y_tr)

auc_honest = roc_auc_score(y_te, m_honest.predict_proba(Xh_te)[:, 1])
auc_leaky  = roc_auc_score(y_te, m_leaky.predict_proba(Xl_te)[:, 1])

print('=== THE LEAKAGE TRAP ===')
print(f'Honest features only, ROC AUC:      {auc_honest:.3f}')
print(f'WITH the leaky column added, ROC AUC: {auc_leaky:.3f}  <-- jumps toward 1.0, this is the trap')
print()
print('Deleting LEAKY_ctr_bucket now and keeping the honest number as the real result.')

# Delete the leaky column -- keep only the honest result going forward
del features['LEAKY_ctr_bucket']
print(f'Honest ROC AUC kept: {auc_honest:.3f}')

=== THE LEAKAGE TRAP ===
Honest features only, ROC AUC:      0.790
WITH the leaky column added, ROC AUC: 0.837  <-- jumps toward 1.0, this is the trap

Deleting LEAKY_ctr_bucket now and keeping the honest number as the real result.
Honest ROC AUC kept: 0.790


## 4. Data limits

**Named limitation: this is an unbalanced panel.** Different clients' tracking history starts on different dates (`dim_clients.gsc_data_start` / `ga4_data_start` differ per client) — so a March 2026 slice does not mean "March data for every client equally." Some clients may have little or no history that far back, and rows before a client's `ga4_data_start` carry search data only (`ga4_data_available = FALSE`), not zero engagement. Treating "no tracking yet" as "no traffic" would be a real mistake this data could easily hide if I didn't check `dim_clients` first — which is exactly why Query 3 checks availability with `IS TRUE` rather than assuming every row is usable.

In [11]:
# Optional: check dim_clients directly to see how much history-start varies per client
clients_check = con.sql(f"""
    SELECT COUNT(*) AS n_clients,
           MIN(gsc_data_start) AS earliest_gsc_start,
           MAX(gsc_data_start) AS latest_gsc_start,
           SUM(CASE WHEN ga4_data_start IS NULL THEN 1 ELSE 0 END) AS clients_with_no_ga4_start
    FROM {TABLES['dim_clients']}
""").df()
print('=== dim_clients: how unbalanced is the panel? ===')
print(clients_check)

=== dim_clients: how unbalanced is the panel? ===
   n_clients earliest_gsc_start latest_gsc_start  clients_with_no_ga4_start
0        104         2025-01-27       2026-06-02                       53.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all) — **run this in Colab with your HF_TOKEN before checking this box**
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.